# SAAM Project — Part I, Section 2.2: Minimum-Variance Portfolio
**Group AT** — North America (AMER) / Scope 1+2

This notebook implements the out-of-sample minimum-variance portfolio with:
1. Rolling 10-year estimation window for expected returns and covariance matrix
2. Long-only constrained optimization (weights ≥ 0, sum to 1)
3. Monthly ex-post performance with buy-and-hold weight drift
4. Summary statistics: annualized return, volatility, Sharpe ratio, min, max

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ──────────────────────────────────────────────────────────
ESTIMATION_WINDOW = 120      # 10 years × 12 months
Y0                = 2013     # first allocation year
Y_END             = 2024     # last allocation year
DATA_DIR          = './cleaned_data/'

print('Setup complete.')

## 1. Load Cleaned Data

In [ ]:
# Returns
returns_m = pd.read_csv(f'{DATA_DIR}returns_monthly.csv', index_col=0, parse_dates=False)
returns_m.columns = pd.to_datetime(returns_m.columns)

# RI monthly (for price-availability checks)
ri_m = pd.read_csv(f'{DATA_DIR}ri_monthly.csv', index_col=0, parse_dates=False)
ri_m.columns = pd.to_datetime(ri_m.columns)

# Market values (monthly, for value-weighted portfolio later)
mv_m = pd.read_csv(f'{DATA_DIR}mv_monthly.csv', index_col=0, parse_dates=False)
mv_m.columns = pd.to_datetime(mv_m.columns)

# Investment sets
inv_sets_df = pd.read_csv(f'{DATA_DIR}investment_sets.csv')
investment_sets = {}
for Y, grp in inv_sets_df.groupby('Year'):
    investment_sets[int(Y)] = grp['ISIN'].tolist()

# Risk-free rate
rf_df = pd.read_csv(f'{DATA_DIR}risk_free_rate.csv')
rf_dict = {(int(r['year']), int(r['month'])): r['RF_monthly']
           for _, r in rf_df.iterrows()}

# Firm names
firm_names = pd.read_csv(f'{DATA_DIR}firm_names.csv', index_col=0).squeeze()

# Return dates
dates_ret = returns_m.columns.tolist()

# December indices in returns
dec_ret_idx = {d.year: i for i, d in enumerate(dates_ret) if d.month == 12}

print(f'Returns: {returns_m.shape[0]} firms × {returns_m.shape[1]} months')
print(f'Investment sets loaded for years {min(investment_sets.keys())}–{max(investment_sets.keys())}')
print(f'Risk-free rate: {len(rf_dict)} months')

## 2. Minimum-Variance Optimization

For each year $Y$ from 2013 to 2024:
1. Select the investment set for year $Y$
2. Extract the 10-year rolling window of monthly returns (120 months ending Dec $Y$)
3. Fill NaN returns with 0
4. Estimate $\hat{\mu}_Y$ (mean) and $\Sigma_Y$ (covariance matrix)
5. Solve: $\min_{\alpha} \alpha' \Sigma_Y \alpha$ s.t. $\alpha' \mathbf{1} = 1$, $\alpha_i \geq 0$
6. Store optimal weights for year $Y+1$

In [ ]:
def solve_min_variance(cov_matrix, n_assets):
    """
    Solve the long-only minimum-variance portfolio problem.
    
    min  alpha' @ Sigma @ alpha
    s.t. sum(alpha) = 1
         alpha_i >= 0  for all i
    """
    # Objective: portfolio variance
    def objective(alpha):
        return alpha @ cov_matrix @ alpha
    
    # Gradient of the objective (for faster convergence)
    def gradient(alpha):
        return 2 * cov_matrix @ alpha
    
    # Constraints
    constraints = {'type': 'eq', 'fun': lambda a: np.sum(a) - 1.0}
    
    # Bounds: 0 <= alpha_i (long-only)
    bounds = [(0, None)] * n_assets
    
    # Initial guess: equal weights
    x0 = np.ones(n_assets) / n_assets
    
    result = minimize(
        objective,
        x0,
        jac=gradient,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'maxiter': 1000, 'ftol': 1e-12}
    )
    
    if not result.success:
        print(f'  ⚠ Optimization warning: {result.message}')
    
    # Clean small negative weights (numerical noise)
    weights = result.x.copy()
    weights[weights < 1e-10] = 0.0
    weights /= weights.sum()  # re-normalize
    
    return weights, result.fun


print('Optimization function defined.')

In [ ]:
# ── Rolling optimization: year by year ──────────────────────────────────

optimal_weights = {}   # {year: pd.Series(weights, index=ISINs)}
estimation_stats = []  # diagnostics

for Y in range(Y0, Y_END + 1):
    # 1. Investment set
    isins_Y = investment_sets[Y]
    n_assets = len(isins_Y)
    
    # 2. Estimation window: 120 months ending at Dec of year Y
    end_idx = dec_ret_idx[Y]
    start_idx = end_idx - ESTIMATION_WINDOW + 1
    window_cols = dates_ret[start_idx : end_idx + 1]
    
    # 3. Extract returns for eligible firms, fill NaN with 0
    ret_window = returns_m.loc[isins_Y, window_cols].fillna(0.0)
    
    # 4. Estimate mean and covariance
    mu_hat = ret_window.mean(axis=1).values            # (N,)
    cov_matrix = ret_window.T.cov().values             # (N, N) — sample covariance
    
    # Note: pandas .cov() uses (tau - 1) denominator (unbiased).
    # The project formula uses 1/tau. We adjust:
    tau = len(window_cols)
    cov_matrix = cov_matrix * (tau - 1) / tau
    
    # 5. Solve min-variance
    weights, opt_var = solve_min_variance(cov_matrix, n_assets)
    
    # 6. Store
    optimal_weights[Y] = pd.Series(weights, index=isins_Y)
    
    # Diagnostics
    n_nonzero = (weights > 1e-8).sum()
    top_weight = weights.max()
    estimation_stats.append({
        'Year': Y,
        'N_firms': n_assets,
        'N_nonzero_weights': n_nonzero,
        'Max_weight': top_weight,
        'Min_var (monthly)': opt_var,
        'Min_vol (ann.)': np.sqrt(opt_var * 12),
    })
    
    print(f'  {Y}: {n_assets} firms, {n_nonzero} non-zero weights, '
          f'max weight = {top_weight:.3f}, ann. vol = {np.sqrt(opt_var*12):.4f}')

df_stats = pd.DataFrame(estimation_stats).set_index('Year')
print('\nDone. Optimal weights computed for all years.')

## 3. Ex-Post Portfolio Returns (Buy-and-Hold Drift)

For each year $Y+1$, compute monthly portfolio returns using the weights determined at end of $Y$, with buy-and-hold drift:

$$R_{p,t+k} = \alpha_{t+k-1}' R_{t+k}$$

$$\alpha_{i,t+k-1} = \frac{\alpha_{i,t+k-2} \times (1 + R_{i,t+k-1})}{1 + R_{p,t+k-1}}$$

In [ ]:
portfolio_returns = []  # list of (date, return)

for Y in range(Y0, Y_END + 1):
    # Weights decided at end of Y, applied in Y+1
    w_Y = optimal_weights[Y]
    isins_Y = w_Y.index.tolist()
    
    # Monthly dates in year Y+1
    year_next = Y + 1
    months_next = [d for d in dates_ret 
                   if d.year == year_next and d.month >= 1 and d.month <= 12]
    
    if len(months_next) == 0:
        continue
    
    # Current weights (will drift)
    alpha_current = w_Y.values.copy()
    
    for dt in months_next:
        # Stock returns for this month (NaN → 0)
        r_stocks = returns_m.loc[isins_Y, dt].fillna(0.0).values
        
        # Portfolio return
        r_port = alpha_current @ r_stocks
        portfolio_returns.append({'Date': dt, 'Return': r_port})
        
        # Update weights (buy-and-hold drift)
        alpha_current = alpha_current * (1 + r_stocks) / (1 + r_port)

# Assemble time series
df_port_ret = pd.DataFrame(portfolio_returns)
df_port_ret['Date'] = pd.to_datetime(df_port_ret['Date'])
df_port_ret = df_port_ret.set_index('Date').sort_index()

print(f'Portfolio return series: {len(df_port_ret)} months')
print(f'Period: {df_port_ret.index[0].strftime("%Y-%m")} to {df_port_ret.index[-1].strftime("%Y-%m")}')
df_port_ret.head()

## 4. Portfolio Performance Statistics

Compute:
- Annualized average return: $\bar{\mu}_p = \hat{\mu}_{p,monthly} \times 12$
- Annualized volatility: $\sigma_p = \hat{\sigma}_{p,monthly} \times \sqrt{12}$
- Sharpe ratio: $SR_p = (\bar{\mu}_p - \bar{r}_f) / \sigma_p$
- Minimum and maximum monthly returns

In [ ]:
# Portfolio returns array
r_p = df_port_ret['Return'].values

# Mean risk-free rate over the portfolio period
rf_values = []
for dt in df_port_ret.index:
    key = (dt.year, dt.month)
    if key in rf_dict:
        rf_values.append(rf_dict[key])
rf_mean_monthly = np.mean(rf_values)

# Statistics
mu_monthly  = np.mean(r_p)
vol_monthly = np.std(r_p, ddof=0)  # population std, consistent with project formula

mu_ann  = mu_monthly * 12
vol_ann = vol_monthly * np.sqrt(12)
rf_ann  = rf_mean_monthly * 12
sharpe  = (mu_ann - rf_ann) / vol_ann

r_min = np.min(r_p)
r_max = np.max(r_p)

print('═══════════════════════════════════════════════════════')
print('  MINIMUM-VARIANCE PORTFOLIO — P(mv)_oos')
print('═══════════════════════════════════════════════════════')
print(f'  Annualized return (μ̄_p):     {mu_ann:>8.4f}  ({mu_ann*100:.2f}%)')
print(f'  Annualized volatility (σ_p):  {vol_ann:>8.4f}  ({vol_ann*100:.2f}%)')
print(f'  Risk-free rate (avg ann.):     {rf_ann:>8.4f}  ({rf_ann*100:.2f}%)')
print(f'  Sharpe ratio:                  {sharpe:>8.4f}')
print(f'  Min monthly return:            {r_min:>8.4f}  ({r_min*100:.2f}%)')
print(f'  Max monthly return:            {r_max:>8.4f}  ({r_max*100:.2f}%)')
print('═══════════════════════════════════════════════════════')

## 5. Cumulative Return Series

In [ ]:
# Cumulative return (wealth index starting at 1)
cumulative_ret = (1 + df_port_ret['Return']).cumprod()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cumulative_ret.index, cumulative_ret.values, color='#1f4e79', linewidth=1.5, label='P(mv)_oos')
ax.set_title('Cumulative Return — Minimum-Variance Portfolio (Long-Only)', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return (wealth index, starting at 1)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('mv_cumulative_return.png', dpi=150)
plt.show()

print(f'Final cumulative return: {cumulative_ret.iloc[-1]:.4f} ({(cumulative_ret.iloc[-1]-1)*100:.2f}%)')

## 6. Weight Analysis

In [ ]:
# Show top 10 holdings for the first and last allocation years
for Y in [Y0, Y_END]:
    w = optimal_weights[Y]
    w_sorted = w[w > 1e-6].sort_values(ascending=False)
    top10 = w_sorted.head(10)
    
    print(f'\nTop 10 holdings at end of {Y} (for {Y+1}):')
    print(f'{"ISIN":<20} {"Name":<35} {"Weight":>8}')
    print('-' * 65)
    for isin, weight in top10.items():
        name = firm_names.get(isin, 'N/A')[:33]
        print(f'{isin:<20} {name:<35} {weight:>8.4f}')
    print(f'Sum of top 10: {top10.sum():.4f}')
    print(f'Total non-zero weights: {(w > 1e-6).sum()}')

## 7. Estimation Diagnostics

In [ ]:
# Summary table of estimation across years
print('Optimization summary across years:\n')
print(df_stats.to_string())

## 8. Save Results

Save portfolio returns and weights for use in subsequent sections.

In [ ]:
# Save portfolio returns
df_port_ret.to_csv(f'{DATA_DIR}mv_portfolio_returns.csv')

# Save optimal weights (one file per year, or a single long-format file)
weights_rows = []
for Y, w in optimal_weights.items():
    for isin, weight in w.items():
        if weight > 1e-10:
            weights_rows.append({'Year': Y, 'ISIN': isin, 'Weight': weight})
df_weights = pd.DataFrame(weights_rows)
df_weights.to_csv(f'{DATA_DIR}mv_optimal_weights.csv', index=False)

# Save summary statistics
summary = pd.DataFrame([{
    'Portfolio': 'P(mv)_oos',
    'Ann_Return': mu_ann,
    'Ann_Volatility': vol_ann,
    'Sharpe_Ratio': sharpe,
    'Min_Monthly_Return': r_min,
    'Max_Monthly_Return': r_max,
    'RF_ann': rf_ann,
}])
summary.to_csv(f'{DATA_DIR}mv_summary_stats.csv', index=False)

print('All results saved.')
print(f'  - {DATA_DIR}mv_portfolio_returns.csv')
print(f'  - {DATA_DIR}mv_optimal_weights.csv')
print(f'  - {DATA_DIR}mv_summary_stats.csv')